# DESC ELAsTiCC2 — SALT2 light-curve fitting with sncosmo

- **author** : Sylvie Dagoret-Campagne
- **affiliation** : IJCLab/IN2P3/CNRS — Université Paris-Saclay
- **creation date** : 2026-05-07
- **last update**: 2026-05-12 : refactor computemustaterrors (gradient g^T.C.g)

## Purpose

This notebook fits SNIa light curves from the ELAsTiCC2 training sample using the
**SALT2** model (`salt2-extended`) provided by `sncosmo`.  
The five SALT2 parameters — `z`, `t0`, `x0`, `x1`, `c` — are fitted **simultaneously
across all six LSST bands** (u, g, r, i, z, y) without any per-band renormalisation.

The SALT2 model encodes the full multi-band spectral energy distribution of SNIa;
no colour factor is needed between bands.

### References
- sncosmo documentation: https://sncosmo.readthedocs.io/en/stable/index.html
- ELAsTiCC2 dataset: DESC TD public data
- Notebook `01_readsnana/03_elasticc2_fit_lightcurves.ipynb` — data-loading pattern
- Notebook `02_sncosmo/01_sncosmo.ipynb` — sncosmo model usage


## 0 · Imports

In [ ]:
%matplotlib inline

import sys
import os
import math
import pathlib
import logging
import warnings

import numpy as np
import pandas as pd
import astropy.table
import matplotlib
from matplotlib import pyplot as plt

import sncosmo

# ── local library ──────────────────────────────────────────────────────────────
libdir = pathlib.Path(os.getcwd()).parent.parent / "lib_elasticc2"
sys.path.insert(0, str(libdir))
from transcientslightcurves import elasticc2_snana_reader

# ── logging ────────────────────────────────────────────────────────────────────
_logger = logging.getLogger("main")
if not _logger.hasHandlers():
    _logout = logging.StreamHandler(sys.stderr)
    _logger.addHandler(_logout)
    _logout.setFormatter(logging.Formatter(
        '[%(asctime)s - %(levelname)s] - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    ))
_logger.setLevel(logging.INFO)
_logger.info("Imports done.")

In [ ]:
try:
    import ipympl  # noqa: F401
    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")
    print("Install with:  pip install ipympl")

## 1 · Parameters

In [ ]:
OBJ_CLASS      = 'SNIa-SALT3'
Z_MIN          = 0.001
Z_MAX          = 1.5
FILE_NUM       = 1
MIN_DETECTIONS = 8
DETECTED_ONLY  = True
N_CURVES       = 1000
RANDOM_SEED    = 42

SALT2_SOURCE   = 'salt2-extended'
ZP             = 31.4
ZPSYS          = 'ab'
BANDS          = ['u', 'g', 'r', 'i', 'z', 'y']
BAND_PREFIX    = 'lsst'

DATA_DIR   = "/Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2"
DIR_PREFIX = "ELASTICC2_TRAIN_02_"

BAND_COLORS = {
    'u': '#cc0ccc', 'g': '#00cc44', 'r': '#cc0000',
    'i': '#ff4400', 'z': '#886600', 'y': '#442200'
}
NCOLS = 4

rng = np.random.default_rng(seed=RANDOM_SEED)
print(f"SALT2 source : {SALT2_SOURCE}")
print(f"ZP={ZP}  zpsys={ZPSYS}")
print(f"Bands : {BANDS}")
print(f"N_CURVES={N_CURVES}")

## 2 · sncosmo model and band registration check

In [ ]:
salt2_model = sncosmo.Model(source=SALT2_SOURCE)
print(f"SALT2 model loaded: {salt2_model}")
print(f"Parameters: {salt2_model.param_names}")

print("\nChecking LSST band registration in sncosmo:")
for b in BANDS:
    bname = BAND_PREFIX + b
    try:
        band_obj = sncosmo.get_bandpass(bname)
        print(f"  {bname:10s}  →  {band_obj}  ✓")
    except Exception as e:
        print(f"  {bname:10s}  →  NOT FOUND: {e}")

## 3 · Load ELAsTiCC2 data

In [ ]:
esr = elasticc2_snana_reader(DATA_DIR, dir_prefix=DIR_PREFIX)

_logger.info(f"Loading HEAD for {OBJ_CLASS}...")
head  = esr.get_head(OBJ_CLASS, return_format='pandas')
_logger.info(f"Loading truth for {OBJ_CLASS}...")
truth = esr.get_object_truth(OBJ_CLASS, return_format='pandas')
_logger.info(f"Loading light curves (file_num={FILE_NUM})...")
all_ltcvs = esr.get_all_ltcvs(OBJ_CLASS, file_num=FILE_NUM, return_format='pandas')
_logger.info("Done.")

print(f"{all_ltcvs['SNID'].nunique()} objects loaded.")
print(f"Columns: {list(all_ltcvs.columns)}")

In [ ]:
detcounts = (
    all_ltcvs[(all_ltcvs['PHOTFLAG'] & esr.photflag_detect) != 0]
    .groupby('SNID').agg('count')['MJD']
    .reset_index()
    .rename({'MJD': 'ndetect'}, axis=1)
)
truth_counts = truth.join(detcounts.set_index('SNID'), on='SNID', how='inner')
subset = truth_counts[
    (truth_counts['ZCMB'] >= Z_MIN) &
    (truth_counts['ZCMB'] <  Z_MAX) &
    (truth_counts['ndetect'] >= MIN_DETECTIONS)
].copy()
print(f"{len(subset)} objects pass selection (z∈[{Z_MIN},{Z_MAX}), ndet≥{MIN_DETECTIONS}).")

## 4 · Helper: build an `astropy.Table` for sncosmo

In [ ]:
def make_sncosmo_table(ltcv_df, detected_only=True,
                       photflag_detect=None,
                       zp=ZP, zpsys=ZPSYS,
                       band_prefix=BAND_PREFIX):
    """Convert an ELAsTiCC2 light-curve DataFrame to an astropy.Table
    suitable for sncosmo.fit_lc.
    """
    df = ltcv_df.copy()
    if detected_only and photflag_detect is not None:
        df = df[(df['PHOTFLAG'] & photflag_detect) != 0]
    df = df[df['FLUXCALERR'] > 0].copy()
    df['BAND'] = df['BAND'].str.strip().str.lower()
    df['band_sncosmo'] = band_prefix + df['BAND']
    return astropy.table.Table({
        'time'    : df['MJD'].values.astype(float),
        'band'    : df['band_sncosmo'].values,
        'flux'    : df['FLUXCAL'].values.astype(float),
        'fluxerr' : df['FLUXCALERR'].values.astype(float),
        'zp'      : np.full(len(df), zp, dtype=float),
        'zpsys'   : np.full(len(df), zpsys),
    })

print("make_sncosmo_table helper ready.")

## 5 · Helper: fit a single event with SALT2

**Free parameters**: `t0`, `x0`, `x1`, `c`.  
**Fixed parameter**: `z` — truth redshift `ZCMB` from ELAsTiCC2.

In [ ]:
def fit_salt2_event(ltcv_df, z_true, fit_z=False, photflag_detect=None):
    """Fit one ELAsTiCC2 SNIa event with SALT2 (sncosmo)."""
    try:
        obs = make_sncosmo_table(ltcv_df, detected_only=DETECTED_ONLY,
                                 photflag_detect=photflag_detect)
    except Exception as e:
        return {'success': False, 'message': f'Table creation failed: {e}'}

    if len(obs) < 5:
        return {'success': False, 'message': 'Not enough data points after filtering'}

    model = sncosmo.Model(source=SALT2_SOURCE)
    t0_guess = obs['time'][np.argmax(obs['flux'])]
    model.set(z=z_true, t0=t0_guess, x0=1e-4, x1=0.0, c=0.0)

    vparam_names = ['t0', 'x0', 'x1', 'c']
    bounds = {
        't0': (t0_guess - 30.0, t0_guess + 30.0),
        'x0': (1e-8, 1.0), 'x1': (-5.0, 5.0), 'c': (-0.5, 0.5),
    }
    if fit_z:
        vparam_names = ['z', 't0', 'x0', 'x1', 'c']
        bounds['z'] = (max(z_true - 0.1, 0.001), z_true + 0.1)

    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            result, fitted_model = sncosmo.fit_lc(
                obs, model, vparam_names=vparam_names,
                bounds=bounds, minsnr=0.0, warn=False
            )
    except Exception as e:
        return {'success': False, 'message': f'Fit failed: {e}', 'table': obs}

    chi2 = result.chisq
    ndof = result.ndof
    return {
        'success': True, 'result': result, 'fitted_model': fitted_model,
        'table': obs,
        'z': fitted_model['z'], 't0': fitted_model['t0'],
        'x0': fitted_model['x0'], 'x1': fitted_model['x1'], 'c': fitted_model['c'],
        'chi2': chi2, 'ndof': ndof, 'chi2_red': chi2 / max(ndof, 1),
        'message': result.message,
    }

print("SALT2 single-event fitter ready.")

## 6 · Select events and run SALT2 fits

In [ ]:
n_avail = min(N_CURVES, len(subset))
chosen_idx   = rng.choice(len(subset), size=n_avail, replace=False)
chosen_snids = subset['SNID'].values[chosen_idx]
print(f"Selected {n_avail} SNIDs for SALT2 fitting.")

In [ ]:
def computemustaterrors(param_names, params, covariance,
                        mb_const=10.635, M=-19.3, alpha=0.14, beta=3.1,
                        cosmo=None):
    """Propagate SALT2 fit covariance into an uncertainty on the Tripp distance modulus.

    Tripp formula
    -------------
        mu  =  m_B*  -  M  +  alpha*x1  -  beta*c
        m_B*  =  -2.5 * log10(x0)  +  mb_const

    Partial derivatives of mu (gradient vector g):
        d(mu)/d(x0) = -2.5 / (x0 * ln10)
        d(mu)/d(x1) =  alpha
        d(mu)/d(c)  = -beta
        d(mu)/d(z)  = (5/ln10)*(1/d_L)*dd_L/dz   [only when z is a free parameter]

    The full error propagation is evaluated as:
        sigma_mu = sqrt( g^T . C_sub . g )
    where C_sub is the covariance sub-matrix restricted to the parameters
    that enter mu (x0, x1, c and optionally z).  t0 is excluded.

    Parameters
    ----------
    param_names : list[str]
        Names of *all* fitted parameters, in the same order as `params` and
        the rows/columns of `covariance`.
        z fixed : ['t0', 'x0', 'x1', 'c']
        z free  : ['z', 't0', 'x0', 'x1', 'c']
    params      : array-like  – fitted values, same order as param_names.
    covariance  : 2-D array   – full covariance matrix, same order.
    mb_const    : SALT2 zero-point offset (default 10.635 for salt2-extended).
    M           : absolute B-band magnitude of a standard SNIa.
    alpha       : Tripp stretch coefficient.
    beta        : Tripp colour coefficient.
    cosmo       : astropy cosmology instance; used only when z is free.
                  Defaults to FlatLambdaCDM(H0=70, Om0=0.3).

    Returns
    -------
    dict with keys:
        mu          – Tripp distance modulus
        sigma_mu    – 1-sigma uncertainty on mu
        gradient    – {param_name: d(mu)/d(param)} for mu-entering params
        sigmas      – {param_name: 1-sigma marginal error} for all params
        covariances – {'pi:pj': cov(pi,pj)} for mu-entering pairs
    """
    params = np.asarray(params, dtype=float)
    cov    = np.asarray(covariance, dtype=float)
    names  = list(param_names)

    # Build a name→index map so we never hard-code positions
    idx = {name: i for i, name in enumerate(names)}

    # Retrieve the three SALT2 photometric parameters by name
    x0 = params[idx['x0']]
    x1 = params[idx['x1']]
    c  = params[idx['c']]

    # Tripp distance modulus
    mB_star = -2.5 * np.log10(x0) + mb_const
    mu      = mB_star - M + alpha * x1 - beta * c

    # Gradient of mu w.r.t. the parameters that enter it
    gradient  = {
        'x0': -2.5 / (x0 * np.log(10)),   # d(m_B*)/d(x0)
        'x1':  alpha,                       # d(mu)/d(x1)
        'c' : -beta,                        # d(mu)/d(c)
    }
    mu_params = ['x0', 'x1', 'c']

    # Add the z contribution only when z was a free parameter
    if 'z' in idx:
        z = params[idx['z']]
        if cosmo is None:
            from astropy.cosmology import FlatLambdaCDM
            cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
        dz   = 1e-5
        dL   = cosmo.luminosity_distance(z).value
        dL_p = cosmo.luminosity_distance(z + dz).value
        dL_m = cosmo.luminosity_distance(z - dz).value
        gradient['z'] = (5.0 / np.log(10)) * (dL_p - dL_m) / (2.0 * dz * dL)
        mu_params.append('z')

    # Gradient vector g and covariance sub-matrix C_sub
    # restricted to the parameters that actually enter mu  (t0 excluded)
    g       = np.array([gradient[p] for p in mu_params])
    sub_idx = np.array([idx[p]      for p in mu_params])
    C_sub   = cov[np.ix_(sub_idx, sub_idx)]

    # sigma_mu^2 = g^T . C_sub . g  (exact first-order error propagation)
    sigma_mu = float(np.sqrt(max(0.0, g @ C_sub @ g)))

    # Marginal 1-sigma errors on every fitted parameter
    sigmas = {name: float(np.sqrt(max(0.0, cov[i, i])))
              for name, i in idx.items()}

    # Off-diagonal covariances for mu-entering pairs (upper triangle only)
    covariances = {}
    for ia, a in enumerate(mu_params):
        for ib, b in enumerate(mu_params):
            if ia < ib:
                covariances[f'{a}:{b}'] = float(C_sub[ia, ib])

    return {
        'mu'         : float(mu),
        'sigma_mu'   : sigma_mu,
        'gradient'   : gradient,
        'sigmas'     : sigmas,
        'covariances': covariances,
    }


print("computemustaterrors ready  (gradient g^T.C.g, param_names-aware)")

In [ ]:
fit_results = {}
fit_errors  = {}

for snid in chosen_snids:
    ltcv   = all_ltcvs[all_ltcvs['SNID'] == snid]
    z_row  = truth[truth['SNID'] == snid]['ZCMB'].values
    z_true = float(z_row[0]) if len(z_row) else np.nan

    res = fit_salt2_event(ltcv, z_true=z_true,
                          fit_z=False, photflag_detect=esr.photflag_detect)
    fit_results[snid] = res

    #param_names = res["result"].param_names
    param_names = res["result"].vparam_names 
    params      = res["result"].parameters
    cov         = res["result"].covariance
    fit_errors[snid] = computemustaterrors(param_names, params, cov)
    sigma_mu = fit_errors[snid]["sigma_mu"]

    if res['success']:
        print(
            f"  SNID {snid:8d}  ✓  "
            f"z={res['z']:.4f}  t0={res['t0']:.2f}  "
            f"x0={res['x0']:.3e}  x1={res['x1']:+.3f}  c={res['c']:+.3f}  "
            f"χ²/dof={res['chi2_red']:.2f}  σ_mu={sigma_mu:.3f}"
        )
    else:
        print(f"  SNID {snid:8d}  ✗  {res['message']}")

In [ ]:
df_errors = pd.DataFrame(fit_errors).T
pd.set_option('display.max_rows', None)
pd.options.display.float_format = '{:.3g}'.format
display(df_errors)

## 7 · Plot: multi-band light curves with SALT2 fit

In [ ]:
def plot_salt2_fit(ax, snid, res, z_true):
    if not res.get('success'):
        ax.set_title(f"SNID {snid}\nFit failed\n{res.get('message','')}",
                     fontsize=8, color='red')
        return
    obs, fitted_model = res['table'], res['fitted_model']
    t0, chi2_red = res['t0'], res['chi2_red']
    t_dense = np.linspace(obs['time'].min() - 10, obs['time'].max() + 10, 400)
    for b in BANDS:
        bname = BAND_PREFIX + b
        color = BAND_COLORS.get(b, 'gray')
        mask = np.array(obs['band']) == bname
        if mask.sum() > 0:
            ax.errorbar(obs['time'][mask] - t0, obs['flux'][mask],
                        yerr=obs['fluxerr'][mask], color=color,
                        linestyle='None', marker='o', markersize=4,
                        capsize=2, label=b, zorder=3)
        try:
            f_model = fitted_model.bandflux(bname, t_dense, zp=ZP, zpsys=ZPSYS)
            valid = np.isfinite(f_model)
            if valid.sum() > 1:
                ax.plot(t_dense[valid] - t0, f_model[valid],
                        color=color, lw=1.5, ls='-', zorder=2)
        except Exception:
            pass
    ax.axhline(0.0, color='k', lw=0.5, ls='--')
    ax.set_title(
        f"SNID {snid}  z={z_true:.3f}\n"
        f"t0={t0:.1f}  x1={res['x1']:+.2f}  c={res['c']:+.2f}  χ²/dof={chi2_red:.2f}",
        fontsize=8)
    ax.set_xlabel(r"$t - t_0$ [days]", fontsize=8)
    ax.set_ylabel("FLUXCAL", fontsize=8)
    ax.tick_params(axis='both', labelsize=7)
    ax.legend(fontsize=6, ncol=3, loc='upper right')

nrows = math.ceil(n_avail / NCOLS)
fig, axes = plt.subplots(nrows, NCOLS,
                          figsize=(5.5 * NCOLS, 4.2 * nrows), tight_layout=True)
axes_flat = np.array(axes).flatten()
for idx, snid in enumerate(chosen_snids):
    z_row  = truth[truth['SNID'] == snid]['ZCMB'].values
    z_true = float(z_row[0]) if len(z_row) else np.nan
    plot_salt2_fit(axes_flat[idx], snid, fit_results[snid], z_true)
for idx in range(n_avail, len(axes_flat)):
    axes_flat[idx].set_visible(False)
fig.suptitle(
    f"SALT2 fits — {OBJ_CLASS}  |  z ∈ [{Z_MIN},{Z_MAX})  |  ndet ≥ {MIN_DETECTIONS}\n"
    f"Model: {SALT2_SOURCE}  |  free params: t0, x0, x1, c  |  z fixed to truth",
    fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## 8 · Summary table of fitted parameters

In [ ]:
rows = []
for snid in chosen_snids:
    res    = fit_results[snid]
    z_row  = truth[truth['SNID'] == snid]['ZCMB'].values
    z_true = float(z_row[0]) if len(z_row) else np.nan
    sigma_mu = fit_errors[snid]["sigma_mu"]
    row = {'SNID': snid, 'z_true': round(z_true, 4), 'success': res.get('success', False)}
    if res.get('success'):
        row.update({
            'z_fit': round(res['z'], 4), 't0': round(res['t0'], 2),
            'x0': float(f"{res['x0']:.4e}"),
            'x1': round(res['x1'], 3), 'c': round(res['c'], 3),
            'chi2': round(res['chi2'], 2), 'ndof': res['ndof'],
            'chi2_red': round(res['chi2_red'], 3),
            'sigma_mu': round(sigma_mu, 3),
        })
    rows.append(row)
df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

## 9 · Distribution of SALT2 parameters

In [ ]:
df_ok = df_results[df_results['success']].copy()
params_plot = ['x1', 'c', 'chi2_red']
labels = {'x1': r'SALT2 $x_1$', 'c': r'SALT2 $c$', 'chi2_red': r'$\chi^2$/dof'}
fig2, axes2 = plt.subplots(1, len(params_plot), figsize=(4.5 * len(params_plot), 3.5),
                            tight_layout=True)
for ax, par in zip(axes2, params_plot):
    vals = df_ok[par].dropna()
    ax.hist(vals, bins=10, color='steelblue', edgecolor='white')
    ax.axvline(np.median(vals), color='k', ls='--', lw=1.5,
               label=f'median={np.median(vals):.3g}')
    ax.set_xlabel(labels[par], fontsize=10)
    ax.set_ylabel('N events', fontsize=10)
    ax.set_title(f'σ = {np.std(vals):.3g}', fontsize=9)
    ax.legend(fontsize=9)
fig2.suptitle(f"SALT2 parameter distributions  —  {OBJ_CLASS}  (N={len(df_ok)} events)",
               fontsize=11)
plt.show()

## 10 · Hubble diagram: distance modulus vs redshift

$$\mu = m_B^* - M_B + \alpha\, x_1 - \beta\, c$$

with $\alpha = 0.14$, $\beta = 3.1$.
Error bars on $\mu$ come from `computemustaterrors` via $\sigma_\mu = \sqrt{\mathbf{g}^T C_{\rm sub}\, \mathbf{g}}$.

In [ ]:
YMIN, YMAX   = 35., 48.
ALPHA_TRIPP  = 0.14
BETA_TRIPP   = 3.10
MB_CONST     = 10.635
MB_CORR_SDC  = +15.

df_ok = df_ok.copy()
df_ok['mB_star']  = -2.5 * np.log10(df_ok['x0'].astype(float)) + MB_CONST
df_ok['mu_tripp'] = (df_ok['mB_star']
                     + ALPHA_TRIPP * df_ok['x1']
                     - BETA_TRIPP  * df_ok['c'] + MB_CORR_SDC)


In [ ]:


fig3, ax3 = plt.subplots(figsize=(8, 6), tight_layout=True)
ax3.errorbar(df_ok['z_true'], df_ok['mu_tripp'],
             yerr=df_ok['sigma_mu'], lw=0, marker='.', color='grey',
             ecolor='grey', capsize=2, elinewidth=1, alpha=0.5)
sc = ax3.scatter(
    df_ok['z_true'], df_ok['mu_tripp'],
    c=df_ok['chi2_red'], cmap='viridis_r', vmin=0, vmax=5,
    s=40, edgecolors='k', linewidths=0.3, zorder=3)
plt.colorbar(sc, ax=ax3, label=r'$\chi^2$/dof')
try:
    from astropy.cosmology import FlatLambdaCDM
    cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
    z_ref = np.linspace(Z_MIN, Z_MAX, 200)
    ax3.plot(z_ref, cosmo.distmod(z_ref).value, 'r--', lw=1.5,
             label=r'Flat $\Lambda$CDM ($H_0$=70, $\Omega_m$=0.3)')
    ax3.legend(fontsize=9)
except Exception:
    pass
ax3.set_xlabel('Redshift z (truth)', fontsize=11)
ax3.set_ylabel(r'$\mu_{\rm Tripp}$ (relative)', fontsize=11)
ax3.set_title(
    f"Hubble diagram — {OBJ_CLASS}  ({len(df_ok)} events)\n"
    rf"$\mu = m_B^* + {ALPHA_TRIPP}\,x_1 - {BETA_TRIPP}\,c$",
    fontsize=11)
ax3.set_ylim(YMIN, YMAX)
ax3.set_xlim(0, Z_MAX)
plt.show()


df_ok['mu_lcdm']  = cosmo.distmod(df_ok['z_true'].values).value
df_ok['mu_resid'] = df_ok['mu_tripp'] - df_ok['mu_lcdm']

fig4, ax4 = plt.subplots(figsize=(8, 4), tight_layout=True)

#plot errorbars
ax4.errorbar(df_ok['z_true'], df_ok['mu_resid'],marker="",
             yerr=df_ok['sigma_mu'], lw=0, color='grey',
             ecolor='grey', capsize=2, elinewidth=0.8, alpha=0.5)

#plot residuals
sc = ax4.scatter(df_ok['z_true'], df_ok['mu_resid'],
                 c=df_ok['chi2_red'], cmap='viridis_r',
                 vmin=0, vmax=5, s=35, edgecolors='k', linewidths=0.3)

ax4.axhline(0.0, color='r', ls='--', lw=1.5)
rms = np.std(df_ok['mu_resid'].dropna())
ax4.set_xlabel('Redshift z (truth)', fontsize=11)
ax4.set_ylabel(r'$\Delta\mu = \mu_{\rm Tripp} - \mu_{\Lambda{\rm CDM}}$', fontsize=10)
ax4.set_title(f"Hubble residuals — SALT2  |  RMS = {rms:.3f} mag", fontsize=11)
ax4.set_ylim(-3.5, 3.5)
ax4.set_xlim(0, Z_MAX)
plt.colorbar(sc, ax=ax4, label=r'$\chi^2$/dof')
plt.show()
print(f"Hubble residual RMS = {rms:.4f} mag  (N={len(df_ok)} events)")

## Summary

| Parameter | Status | Description |
|-----------|--------|-------------|
| `z`  | fixed | truth redshift from ELAsTiCC2 catalogue |
| `t0` | free  | time of B-band maximum (MJD) |
| `x0` | free  | overall amplitude (proportional to luminosity) |
| `x1` | free  | SALT2 stretch (light-curve width) |
| `c`  | free  | SALT2 colour (SED tilt) |

### `computemustaterrors` — design

The function uses `param_names` to build an `idx` dict (`name → position in
covariance`).  All parameter lookups go through `idx`, so the function works
identically whether `z` is fixed (absent from the covariance) or free
(present with its own row/column).  The propagation is the exact
quadratic form  $\sigma^2_\mu = \mathbf{g}^T C_{\rm sub}\, \mathbf{g}$,
computed via NumPy's `@` operator on the sub-matrix restricted to
$\{x_0, x_1, c, [z]\}$ — `t0` is irrelevant and excluded automatically.
